# Eksperimen kalibrasi bridge–chorus

Buka dari root **SongMap**, lalu **Run All**. Python dijalankan dari `.venv-songformer311` di sibling **Section-Annotations**, walaupun kernel notebook berbeda.

Memakai seluruh prediksi train yang sudah disimpan oleh `train_bridge_audit.ipynb`. Kalibrator kecil belajar dari label manual train; bobot SongFormer, section, dan anotasi tidak diubah. Fitur kalibrator: rasio probabilitas bridge/chorus, posisi relatif section, serta durasinya. Hanya prediksi awal bridge/chorus yang dapat ditukar.

Tiga fold berdasarkan grup lagu memilih antara tanpa perubahan (identity) dan tiga kekuatan regularisasi. Pemilihan hanya memakai train. Model SongFormer dasarnya sudah pernah dilatih pada seluruh train: fold ini hanya memisahkan fitting kalibrator, bukan fitting SongFormer. Ini bukan perkiraan generalisasi mandiri.

Setelah pilihan dikunci, satu kandidat dibandingkan dengan baseline pada **32 validation** menggunakan cache encoder asli. Tidak memakai test dan tidak menjalankan encoder lagi. Hasil ini belum mengukur pipeline Core ML terkompresi di aplikasi.


In [ ]:
from pathlib import Path
from datetime import datetime
import json, subprocess

SONGMAP_ROOT=Path.cwd().resolve()
assert (SONGMAP_ROOT/'scripts/calibrate_bridge_chorus.py').is_file(), 'Buka notebook dari root SongMap'
SONGSTRUCT_ROOT=(SONGMAP_ROOT/'../Section-Annotations').resolve()
RUN_PYTHON=SONGSTRUCT_ROOT/'.venv-songformer311/bin/python'
assert RUN_PYTHON.is_file(), f'Python tidak ditemukan: {RUN_PYTHON}'

# Jika audit ada di lokasi lain, ganti TRAIN_AUDIT dengan Path('/lokasi/train-all.json').
files=list((SONGMAP_ROOT/'outputs.local').glob('train-bridge-audit-*/train-all.json'))
assert files, 'Jalankan seluruh train_bridge_audit.ipynb dahulu, sampai train-all.json tersedia.'
TRAIN_AUDIT=max(files,key=lambda p:p.stat().st_mtime).resolve()
audit=json.loads(TRAIN_AUDIT.read_text())
assert audit.get('full_train') and audit.get('split')=='train', 'Audit train belum lengkap'
assert Path(audit['checkpoint']).is_file(), 'Checkpoint audit tidak ditemukan'
DEVICE='mps'  # Harus sama dengan device saat audit train dibuat.
output=SONGMAP_ROOT/'outputs.local'/('bridge-calibration-'+datetime.now().strftime('%Y%m%d-%H%M%S'))
subprocess.run([str(RUN_PYTHON),'-c',
    "import sys,numpy,torch; print('Python:',sys.executable); print('NumPy:',numpy.__version__,'Torch:',torch.__version__)"],check=True)
print('Audit:',TRAIN_AUDIT)
print('Train:',audit['tracks'],'lagu;',audit['eligible_sections'],'section manual')
print('Bridge → chorus:',audit['bridge_to_chorus_count'])
print('Output:',output)


Jalankan cell berikut. Fitting kalibrator memakai tabel prediksi kecil di CPU; hanya forward label model pada validation yang memakai MPS. Prediksi validation disimpan per lagu agar bisa dilanjutkan jika dihentikan. Jika proses terputus, jalankan ulang cell berikut dengan variabel `output` yang sama; tidak perlu mengulang audit train.


In [ ]:
def run_experiment():
    output.mkdir(parents=True,exist_ok=True)
    if (output/'comparison.json').is_file():
        print('Eksperimen sudah selesai:',output/'comparison.json')
        return
    command=[str(RUN_PYTHON),'-u',str(SONGMAP_ROOT/'scripts/calibrate_bridge_chorus.py'),
        '--songstruct-root',str(SONGSTRUCT_ROOT),'--train-audit',str(TRAIN_AUDIT),
        '--device',DEVICE,'--output-dir',str(output)]
    log=output/'experiment.log'
    with log.open('a',encoding='utf-8') as handle:
        process=subprocess.Popen(command,cwd=SONGMAP_ROOT,stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='');handle.write(line);handle.flush()
            status=process.wait()
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    if status:
        raise RuntimeError(f'Gagal ({status}); lihat {log}. Cache validation yang selesai tetap tersedia.')

run_experiment()


In [ ]:
report=json.loads((output/'comparison.json').read_text())
print('Dipilih hanya dari train:',report['adapter_selected_by_train_only'])
print('Train group-CV (bukan skor held-out SongFormer):')
for candidate in report['selection']['candidates']:
    score=candidate['out_of_fold_metrics']
    print(f"  {candidate['candidate']}: F1={score['macro_f1_present']:.4f}")
print('\nValidation:',report['tracks'],'lagu;',report['baseline']['sections'],'section')
for name in ('baseline','candidate'):
    score=report[name]
    print(f"{name}: accuracy={score['section_accuracy']:.2%}; macro F1={score['macro_f1_present']:.4f}")
print('F1 per kelas (baseline → kandidat):')
for label in report['baseline']['label_order']:
    before=report['baseline']['per_class'][label]
    after=report['candidate']['per_class'][label]
    print(f"  {label}: {before['f1']:.4f} → {after['f1']:.4f}; support={before['support']}")
changes=report['changed_sections']
print('Section berubah:',len(changes),
    '| salah → benar:',sum(not c['baseline_correct'] and c['candidate_correct'] for c in changes),
    '| benar → salah:',sum(c['baseline_correct'] and not c['candidate_correct'] for c in changes))
print('Kirim file:',output/'comparison.json')
print('Belum diterapkan ke app. Identity terpilih berarti kalibrasi tidak memberi peningkatan CV train.')


Bagikan **comparison.json**. Laporan mencatat seluruh kelas dan section yang berubah, bukan hanya kenaikan bridge. `adapter.json` menyimpan parameter eksperimen terpisah dari checkpoint.

Probabilitas kalibrator hanya berarti bridge dibanding chorus, bukan vektor probabilitas delapan kelas yang baru. Boundary tetap sama. Peningkatan validation, jika ada, masih perlu diperiksa sebelum menguji kalibrator pada pipeline aplikasi; notebook tidak memasangnya secara otomatis.
